### 🧬 Multiple Sequence Alignment (MSA) Analysis with Biopython
Conserved / variable positions, consensus sequence, and nucleotide frequencies.

#### 1️⃣ Install Biopython

In [1]:
# Install Biopython (use %pip so it installs into the notebook's own kernel environment)
%pip install biopython

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


#### 2️⃣ Import libraries

In [2]:
# MultipleSeqAlignment: container that holds several aligned sequences
from Bio.Align import MultipleSeqAlignment

# Seq: represents a biological sequence (DNA / RNA / protein)
from Bio.Seq import Seq

# SeqRecord: wraps a Seq together with metadata such as its ID
from Bio.SeqRecord import SeqRecord

# Counter: counts how many times each item appears (used for base frequencies)
from collections import Counter

#### 3️⃣ Create the sequences
All sequences must have the **same length** (they are already aligned).

In [29]:
# Create three DNA sequences as SeqRecord objects, each with a unique ID
records = [
    SeqRecord(Seq('ATGCT'), id='Seq1' , name = "11111111"),
    SeqRecord(Seq('ATGTT'), id='Seq2'),
    SeqRecord(Seq('ATGCC'), id='Seq3')
]

print(MultipleSeqAlignment(records)[:])

Alignment with 3 rows and 5 columns
ATGCT Seq1
ATGTT Seq2
ATGCC Seq3


#### 4️⃣ Build and display the alignment

In [12]:
# Combine the records into one alignment object
alignment = MultipleSeqAlignment(records)

# Print the alignment (rows = sequences, columns = positions)
print(alignment)

Alignment with 3 rows and 5 columns
ATGCT Seq1
ATGTT Seq2
ATGCC Seq3


#### 5️⃣ Show sequences position by position

In [50]:
# Print the column numbers (1-based, so humans can read them easily)
print(f"Position: ", " ".join(str(i) for i in range(1, alignment.get_alignment_length() + 1)))
 
 
# print(type(alignment.get_alignment_length())) output: int  

# Print every sequence with its bases separated by spaces
for r in alignment:
    print(r.id + ":", " ".join(r.seq))

Position:  1 2 3 4 5
Seq1: A T G C T
Seq2: A T G T T
Seq3: A T G C C


#### 6️⃣ Conserved vs Variable positions
- **Conserved** → every sequence has the *same* base in that column
- **Variable** → at least one sequence differs

In [51]:
# Lists to store column numbers
conserved_positions = []
variable_positions = []

# Loop through each column of the alignment
for i in range(alignment.get_alignment_length()):
    # Collect the base at column i from every sequence
    column = [str(record.seq[i]) for record in alignment]

    # set() removes duplicates -> only 1 unique base means the column is conserved
    if len(set(column)) == 1:
        conserved_positions.append(i + 1)  # +1 for 1-based numbering
    else:
        variable_positions.append(i + 1)

print("Conserved Positions:", conserved_positions)
print("Variable Positions:", variable_positions)

Conserved Positions: [1, 2, 3]
Variable Positions: [4, 5]


#### 7️⃣ Percentage of conserved positions

In [ ]:
# Total number of columns in the alignment
total_positions = alignment.get_alignment_length()  #5

# Conserved percentage = conserved columns / total columns * 100
conserved_percentage = (len(conserved_positions) / total_positions) * 100

# Format to 2 decimal places and add a % sign
print(f"Conserved percentage: {conserved_percentage:.2f}%")

Conserved percentage: 60.00%


#### 8️⃣ Consensus sequence
The consensus takes the **most common base** in every column.

In [8]:
consensus = ""

for i in range(alignment.get_alignment_length()):
    # Collect all bases in column i
    column = [str(record.seq[i]) for record in alignment]

    # Count each base, e.g. Counter({'C': 2, 'T': 1})
    counts = Counter(column)

    # most_common(1) returns [(base, count)] for the top base
    # (if there is a tie, the base seen first in the column wins)
    most_common_base, count = counts.most_common(1)[0]
    consensus += most_common_base

print("Consensus Sequence:", consensus)

Consensus Sequence: ATGCT


#### 9️⃣ Nucleotide frequencies per position
Count of each base (A, T, G, C) in every column, plus its percentage.

In [9]:
# Bases we want to report, in a fixed order
bases = ["A", "T", "G", "C"]
num_seqs = len(alignment)   # number of rows (sequences)

# Header of the table
print(f"{'Pos':<5}" + "".join(f"{b:>12}" for b in bases))

# Store the frequencies for later use
position_frequencies = {}

for i in range(alignment.get_alignment_length()):
    # Bases in column i from every sequence
    column = [str(record.seq[i]) for record in alignment]
    counts = Counter(column)

    # Percentage of each base in this column
    # (counts[b] returns 0 if the base is absent)
    freqs = {b: (counts[b] / num_seqs) * 100 for b in bases}
    position_frequencies[i + 1] = freqs

    # Print as "count (percentage)"
    row = "".join(f"{f'{counts[b]} ({freqs[b]:.1f}%)':>12}" for b in bases)
    print(f"{i + 1:<5}{row}")

Pos             A           T           G           C
1      3 (100.0%)    0 (0.0%)    0 (0.0%)    0 (0.0%)
2        0 (0.0%)  3 (100.0%)    0 (0.0%)    0 (0.0%)
3        0 (0.0%)    0 (0.0%)  3 (100.0%)    0 (0.0%)
4        0 (0.0%)   1 (33.3%)    0 (0.0%)   2 (66.7%)
5        0 (0.0%)   2 (66.7%)    0 (0.0%)   1 (33.3%)


#### 🔟 Overall base composition (whole alignment)

In [10]:
# Join all sequences into one long string to count every base at once
all_bases = "".join(str(record.seq) for record in alignment)
overall_counts = Counter(all_bases)
total_bases = len(all_bases)

# Percentage of each base across the entire alignment
print("Overall nucleotide composition:")
for b in bases:
    pct = (overall_counts[b] / total_bases) * 100
    print(f"  {b}: {overall_counts[b]:>2} / {total_bases} = {pct:.2f}%")

# GC content = (G + C) / total bases * 100
gc_content = ((overall_counts["G"] + overall_counts["C"]) / total_bases) * 100
print(f"\nGC content: {gc_content:.2f}%")

Overall nucleotide composition:
  A:  3 / 15 = 20.00%
  T:  6 / 15 = 40.00%
  G:  3 / 15 = 20.00%
  C:  3 / 15 = 20.00%

GC content: 40.00%
